# 📘 코드 해설판 (Python 초보자용)

이 노트북은 원본 `2. RAG.ipynb` 에 **번호(`# 1`, `# 2` …)가 붙은 코드 셀마다 한국어 해설**을 덧붙인 파일입니다.
코드 셀 바로 **위**의 `💡 [# N] 코드 설명` 칸이 해설입니다. 원본 코드와 원본 설명은 그대로입니다.

**이 노트북의 흐름**

1. **LLM만 썼을 때의 약점 확인** (# 3 ~ # 12) : 최신 정보, 복잡한 질문, 모호한 질문에 약함
2. **RAG 시제품 만들기** (# 13 ~ # 22) : 도시 100곳의 위키백과 문서를 넣고, 검색해서 답하게 만들기
3. **RAG 개선하기** (# 23 ~ # 36) : 조각 크기, 검색 개수, 프롬프트를 바꿔 가며 더 나은 구성 찾기

> **RAG(Retrieval-Augmented Generation)** = 답하기 전에 관련 자료를 **찾아보고(Retrieval)**, 그 자료를 **참고해 답을 생성(Generation)** 하는 방식. "오픈북 시험"이라고 생각하면 쉽습니다.

#📓 RAG (Retrieval-Augmented Generation) Practice

This is today's main section! In this class, we delve into RAG, the Retrieval-Augmented Generation (RAG) technique. RAG is the process of optimizing the output of a large language model, so it references an external knowledge base outside of its training data sources before generating a response.

Retrieval-Augmented Generation combines the power of language models with the specificity of retrieved documents to generate more accurate and relevant responses. This technique retrieves relevant documents or passages based on the input query and uses this retrieved information to inform the generation process of the language model.  

Let's look at the picture below and remind the structure of the RAG.
<img src="https://i.imgur.com/bR4xaBd.png">
###Steps in RAG
1. The model receives an input query to which a response is required.
2. It retrieves relevant documents or passages from a database that are pertinent to the input query.
3. The language model then generates a response, informed by both the original query and the retrieved documents.


![](https://huggingface.co/blog/assets/12_ray_rag/rag_gif.gif)



\This practice class will be comprised of Two sections.  
  
### I. Building a Prototype RAG.  
### II. Finding the best RAG App Configuration.
  
Okay. Now we know what we have to do.  
Then, is it okay to dive into practice?

![](https://media.tenor.com/r2l6ol9HRqIAAAAj/question-mark-question.gif)

## 0. Before Dive into Practice!

We can easily infer that models often give out strange answers to complex questions. However, this is only natural because people are the same.

So exactly what kind of questions does LLM generate the wrong answers in? If we find out what kind of questions LLM is vulnerable to, we will have a better understanding of how to supplement LLM.

<br/>

We follow the steps below:  

#### 1. Preparing for OpenAI API
#### 2. Make a Method for Asking Question
#### 3. Ask to LLM


### 1. Preparing for OpenAI API
  
We will use OpenAI LLM for this acitivity. So, we need to install openai api library, import it, and set openai api key.

Copy the api key you have to the 'sk-...' location. Note that all api keys start with 'sk-'.

### 💡 [# 1] 코드 설명 — 라이브러리 설치

**한 줄 요약:** 이 실습에 필요한 라이브러리들을 설치합니다.

- `os.environ.pop("PYTHONPATH", None)` : 다른 경로의 라이브러리와 충돌하지 않도록 `PYTHONPATH` 환경변수를 지웁니다.
- `!pip install ...` : `!` 는 터미널 명령 실행 기호입니다. 줄 끝의 `\` 는 "명령이 다음 줄로 이어진다"는 표시입니다.
- 주요 라이브러리
  - `llama-index` : RAG 프레임워크
  - `llama-index-readers-wikipedia`, `wikipedia` : 위키백과 문서를 불러오는 도구
  - `openai` : OpenAI API
  - `nltk` : 문장 나누기 등 언어 처리 도구
  - 나머지(`trulens`, `langchain`, `streamlit` 등)는 평가·화면용 부가 도구로, 이 노트북에서 직접 쓰지는 않습니다.
- `--quiet` : 설치 메시지 최소화

In [2]:
# 1
### YOUR CODE HERE ###
import os
os.environ.pop("PYTHONPATH", None)

!pip install llama-index==0.12.2 llama-index-llms-openai==0.3.1 \
    llama-index-embeddings-huggingface llama-index-readers-wikipedia==0.3.0 wikipedia==1.4.0 \
    openai==1.55.3 trulens==1.2.0 trulens-providers-openai==1.2.0 \
    langchain nltk>=3.8.1 \
    streamlit==1.35.0 watchdog kubernetes==26.1.0 \
    packaging>=24.0 --quiet

### 💡 [# 2] 코드 설명 — API 키 등록과 도구 불러오기

**한 줄 요약:** OpenAI API 키를 등록하고 필요한 도구들을 불러옵니다.

- `os.environ["OPENAI_API_KEY"] = "sk-..."` : API 키를 환경변수에 저장 (본인 키로 교체). **키는 남에게 공유하면 안 됩니다.**
- `VectorStoreIndex` : 검색용 색인 / `Document` : 문서 그릇 / `get_response_synthesizer` : 답변 합성기
- `WikipediaReader` : 위키백과 문서를 읽어 오는 도구
- `SentenceSplitter` : 문서를 문장 단위로 조각내는 도구
- `textwrap` : 긴 글 줄바꿈 도구
- `nltk.download('punkt')` : 문장 분리에 필요한 데이터 파일을 내려받습니다(처음 한 번).

In [3]:
# 2
### YOUR CODE HERE ###

import os
import openai

os.environ["OPENAI_API_KEY"] = "sk-..." #copy your api key

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, get_response_synthesizer
from llama_index.readers.wikipedia import WikipediaReader
from llama_index.core.node_parser import SentenceSplitter

import textwrap

import nltk
nltk.download('punkt')

C:\Users\swsuser-j29\.conda\envs\rag\lib\site-packages\pydantic\_internal\_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\swsuser-j29\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

### 2. Make a Method for Asking Question
  
Using the OpenAI API, we will create a function that receives a question and returns the response of the OpenAI LLM.


### 💡 [# 3] 코드 설명 — LLM에게 질문하는 함수 만들기

**한 줄 요약:** 질문을 넣으면 ChatGPT의 답을 돌려주는 함수 `generate_answer` 를 정의합니다.

- `def generate_answer(question):` : 함수 정의. 실행해도 결과가 보이지 않는 것이 정상입니다.
- `messages` : 대화 목록(리스트). 각 항목은 `{"role": 역할, "content": 내용}` 형태의 사전입니다.
  - `"system"` : AI에게 주는 기본 역할 설정("당신은 도움이 되는 비서입니다")
  - `"user"` : 사용자의 질문
- `openai.chat.completions.create(model=..., messages=...)` : OpenAI 서버에 요청
- `return response.choices[0].message.content` : 응답에서 답변 글만 꺼내 돌려줌

In [6]:
# 3
### YOUR CODE HERE ###

def generate_answer(question):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
    )

    return response.choices[0].message.content

### 3. Ask to LLM

Let's pose some questions to the OpenAI LLM. We will ask the following types of questions:

#### 1. Question about **old information** vs Question about **recent information**
#### 2. Question that **require only simple information** vs Question that **require complex reasoning**
#### 3. Question that are **too vague** vs Question that **clearly specify the desired task**


<br/>

#### 1. Question about **old information** vs Question about **recent information**

Question about old information: 'When did Blackpink make their debut?
'  
Answer: 2016

Question about recent information: 'What is the lowest pressure at which diamonds have been created in the world?'  
Answer: 1 atmospheric pressure

<img src="https://i.imgur.com/ZaHVekw.png" width="600">

<br/>

#### 2. Question that **require only simple information** vs Question that **require complex reasoning**

Question that require only simple information: 'What was the first animal to orbit the Earth?'  
Answer: Laika

Question that require complex reasoning: 'In what country was the current World\'s Strongest Man born?'  

Reasoning step:  
1. Who is the current World\'s Strongest Man?
2. Which country was the current World\'s Strongest Man born in?

Answer: Canada  

<br/>

#### 3. Question that are **too vague** vs Question that **clearly specify the desired task**

Question that are too vague: 'I want to cook something.'  

Question that clearly specify the desired task: 'Explain the steps to make apple pie step by step'

Answer: Compare two responses by yourself  


### 💡 [# 4] 코드 설명 — 실험용 질문 6개 준비

**한 줄 요약:** LLM의 약점을 확인하기 위해 성격이 다른 질문 세 쌍을 변수에 저장합니다.

| 변수 | 성격 |
|---|---|
| `question_old` / `question_recent` | 오래된 정보 vs **최신** 정보 |
| `question_simple` / `question_complex` | 단순 사실 vs **여러 단계 추론**이 필요한 질문 |
| `question_clear` / `question_vague` | 명확한 요청 vs **모호한** 요청 |

- `'World\'s'` : 작은따옴표 문자열 안에서 작은따옴표를 쓰려면 앞에 `\` 를 붙입니다.

In [7]:
# 4
### YOUR CODE HERE ###

question_old =  'When did Blackpink make their debut? '
question_recent = 'What is the lowest pressure at which diamonds have been created in the world?'

question_simple = 'What was the first animal to orbit the Earth?'
question_complex = 'In what country was the current World\'s Strongest Man born?'

question_clear = 'Explain the steps to make apple pie step by step.'
question_vague = 'I want to cook something.'

### 4. Visualizaing Result

Let's check the results by adding a question to the previously made `generate_answer` function.

Also, we will use `pretty_print` method to automatically change the line at a certain length in case the answer is too long.

### 💡 [# 5] 코드 설명 — 보기 좋게 줄바꿈하는 함수

**한 줄 요약:** 긴 답변을 80자 폭으로 줄바꿈해 주는 함수 `pretty_print` 를 만듭니다.

- `textwrap.fill(answer, width=80)` : 글이 한 줄에 80자를 넘지 않게 줄바꿈
- `return wrapped_text` : 정리된 글을 돌려줌 (이름과 달리 직접 출력하지는 않으므로 `print(pretty_print(...))` 처럼 씁니다)

In [8]:
# 5
### YOUR CODE HERE ###

import textwrap

def pretty_print(answer):
    wrapped_text = textwrap.fill(answer, width=80)
    return wrapped_text

### 💡 [# 6] 코드 설명 — 오래된 정보 질문

**한 줄 요약:** "블랙핑크 데뷔 연도"를 LLM에 묻고 실제 정답(2016)과 비교합니다.

- `answer_old = generate_answer(question_old)` : LLM 답변 받기
- `print("문장", 값)` : 쉼표로 구분하면 이어서 출력됩니다.
- 마지막 `print("Real Answer: 2016")` : 비교용 정답

> 🔰 오래전부터 알려진 정보는 LLM이 잘 맞힙니다.

In [9]:
# 6
### YOUR CODE HERE ###

answer_old = generate_answer(question_old)
print("Answer to question with old information: \n", pretty_print(answer_old))
print("------------------------------------------------------------")
print("Real Answer: 2016")

Answer to question with old information: 
 Blackpink made their debut on August 8, 2016.
------------------------------------------------------------
Real Answer: 2016


### 💡 [# 7] 코드 설명 — 최신 정보 질문

**한 줄 요약:** 최근 연구 결과(1기압에서 다이아몬드 합성)를 묻습니다.

- 구조는 # 6과 같고 질문만 다릅니다.

> 🔰 LLM은 **학습이 끝난 시점 이후의 일은 모릅니다.** 그래서 틀리거나 옛 정보를 답할 수 있습니다. → RAG가 필요한 첫 번째 이유

In [10]:
# 7
### YOUR CODE HERE ###

answer_recent = generate_answer(question_recent)
print("Answer to question with recent information: \n", pretty_print(answer_recent))
print("------------------------------------------------------------")
print("Real Answer: 1 atmospheric pressure")

Answer to question with recent information: 
 Diamonds have been created at extreme pressures of about 20 to 60 gigapascals
(GPa), which is equivalent to around 200,000 to 600,000 times Earth's
atmospheric pressure at sea level. These high pressures are typically found deep
within the Earth's mantle or through the use of high-pressure experimental
methods in laboratories.
------------------------------------------------------------
Real Answer: 1 atmospheric pressure


### 💡 [# 8] 코드 설명 — 단순한 질문

**한 줄 요약:** "지구 궤도를 처음 돈 동물"(정답: 라이카)을 묻습니다.

- 구조는 # 6과 동일. 사실 하나만 알면 되는 질문이라 대체로 잘 맞힙니다.

In [11]:
# 8
### YOUR CODE HERE ###

answer_simple = generate_answer(question_simple)
print("Answer to a simple question: \n", pretty_print(answer_simple))
print("------------------------------------------------------------")
print("Real Answer: Laika")

Answer to a simple question: 
 The first animal to orbit the Earth was Laika, a female stray dog from Moscow.
Laika was launched into space on November 3, 1957, aboard the Soviet spacecraft
Sputnik 2. She became the first living being to orbit the Earth.
------------------------------------------------------------
Real Answer: Laika


### 💡 [# 9] 코드 설명 — 복잡한 질문

**한 줄 요약:** "현재 세계에서 가장 힘센 사람이 태어난 나라"를 묻습니다.

- 이 질문은 ① 현재 우승자가 누구인지 → ② 그 사람의 출생지, **두 단계**를 거쳐야 합니다.

> 🔰 "현재"라는 최신 정보까지 섞여 있어 LLM이 틀리기 쉽습니다.

In [12]:
# 9
### YOUR CODE HERE ###

answer_complex = generate_answer(question_complex)
print("Answer to a complex question: \n", pretty_print(answer_complex))
print("------------------------------------------------------------")
print("Real Answer: Canada")

Answer to a complex question: 
 The current World's Strongest Man, Tom Stoltman, was born in Scotland.
------------------------------------------------------------
Real Answer: Canada


### 💡 [# 10] 코드 설명 — 명확한 요청

**한 줄 요약:** "애플파이 만드는 법을 단계별로 설명해 줘"라고 요청합니다.

- 요청이 구체적이므로 답도 목적에 맞게 나옵니다. 마지막 `print` 는 결과를 스스로 평가해 보라는 안내 문구입니다.

In [13]:
# 10
### YOUR CODE HERE ###

answer_clear = generate_answer(question_clear)
print("Response to a clear question: \n", pretty_print(answer_clear))
print("------------------------------------------------------------")
print("\nThink about this results. Does the result fit the purpose of the question?")

Response to a clear question: 
 Sure! Here are the steps to make apple pie:  1. **Prepare the pie crust**:    -
You can make your own pie crust or use a pre-made one. If making your own,
combine flour, salt, and cold butter in a bowl. Cut the butter into the flour
mixture until it resembles coarse crumbs. Gradually add cold water and mix until
the dough comes together. Form it into a disk, wrap in plastic wrap, and chill
in the refrigerator for at least 30 minutes.  2. **Prepare the filling**:    -
Peel, core, and slice apples. Toss them with lemon juice, sugar, cinnamon,
nutmeg, and a pinch of salt. You can also add a little flour or cornstarch to
help thicken the filling.  3. **Assemble the pie**:    - Roll out the chilled
dough on a lightly floured surface to fit your pie dish. Place the crust in the
dish, trim any excess dough hanging over the edges.    - Pour the apple filling
into the crust, spreading it out evenly.    - If you like, you can dot the
filling with small pieces of b

### 💡 [# 11] 코드 설명 — 모호한 요청

**한 줄 요약:** "뭔가 요리하고 싶어"라는 막연한 말을 던져 봅니다.

- 무엇을 원하는지 불분명하면 LLM도 일반적인 답이나 되묻는 답을 합니다.

> 🔰 **질문(프롬프트)의 명확성**이 답의 품질을 좌우한다는 점을 보여 줍니다.

In [14]:
# 11
### YOUR CODE HERE ###

answer_vague = generate_answer(question_vague)
print("Response to a vague question: \n", pretty_print(answer_vague))
print("------------------------------------------------------------")
print("\nThink about this results. Does the result fit the purpose of the question?")

Response to a vague question: 
 That's great! What type of dish are you planning to cook? Do you have any
ingredients in mind or a specific recipe you'd like to try? I can help you with
cooking tips, recipe ideas, and any other advice you may need. Just let me know
how I can assist you further!
------------------------------------------------------------

Think about this results. Does the result fit the purpose of the question?


All right, we've now seen what kind of question LLM is vulnerable to and the importance of a clear prompt.

Finally, before diving into the practice, let's ask LLM complex questions related to the city. We will look at LLM's answers and compare them with those of LLM reinforced by RAG.  

<br/>

Question: **Which Korean city has a relationship with Dresden?**.   
Answer: **Daejeon**

<br/>

<img src="https://media.tenor.com/TWMxi0kGDTgAAAAi/hmm.gif" width="150" />  


Hmm... I think it is quite complex question. Could you answer this question?  
If you can, feel free to think of other questions related to the city.  

<br/>

However, make sure there is a Wikipedia page for that city, and if so, save the title of that Wikipedia page in the variable below.  


### 💡 [# 12] 코드 설명 — RAG로 다시 풀어볼 질문 기억해 두기

**한 줄 요약:** "드레스덴과 관계(자매결연)가 있는 한국 도시는?"을 LLM에 물어 답을 확인해 둡니다.

- `kr_city = generate_answer(city_question)` : RAG **없이** 받은 답
- `city_related_to_question = 'Dresden'` : 나중에 위키백과에서 드레스덴 문서도 꼭 불러오도록 도시 이름을 저장

> 🔰 이 답을 기억해 두세요. # 21에서 RAG를 쓴 답과 비교합니다.

In [15]:
# 12
### YOUR CODE HERE ###

city_question = 'Which Korean city has a relationship with Dresden?'

kr_city = generate_answer(city_question)
print("Answer: ", pretty_print(kr_city))

city_related_to_question = 'Dresden'

Answer:  The Korean city that has a relationship with Dresden is Suwon. Suwon and Dresden
have been sister cities since 1987, fostering cultural exchanges and cooperation
between the two cities.


If you remembered the answer above, let's start making RAG!

## I. Building a Prototype RAG.  

In this section, we will build the Retriever and Database for use in RAG using the LlamaIndex and Milvus that we learned in the previous session.

What makes it different from previous practice is that this time, we go through the process of directly selecting the information and the external knowledge source and inserting it into the database. In this practice, we will make the following assumptions.   
<br/>

#### Input Prompt: **City-related question**  
#### External Knowledge Source: **Wikipedia**   
<br/>

We follow the steps below:  

#### 1. Data Load
#### 2. Check Query Engine
#### 3. Connect Retriever and Generator

### 1. Data Load

Load 100 different cities' wikipedia page data.  

We will use LlamaIndex data connector `WikipediaReader` which that receives a tile list of the wikipedia page and returns corresponding wikipedia page by replacing it with a `Document` object.  

Plus, in this section, we will add the city name to the list corresponding to the city-related questions we initially asked LLM.

### 💡 [# 13] 코드 설명 — 현재 작업 폴더 확인

**한 줄 요약:** 노트북이 실행 중인 폴더 경로를 출력합니다.

- `os.getcwd()` : 현재 작업 폴더. 다음 셀의 파일 경로를 맞게 적기 위한 확인용입니다.

In [15]:
# 13
import os
print(os.getcwd())

C:\Users\RAG\1일차\실습 자료\Code


### 💡 [# 14] 코드 설명 — 도시 이름 목록 파일 읽기

**한 줄 요약:** `city_short.txt` 에서 도시 이름 100개를 읽어 리스트로 만들고, 드레스덴이 없으면 추가합니다.

- `city_name_path = r'...'` : 파일 경로(**본인 경로로 수정**). `r` 은 `\` 를 그대로 쓰기 위한 표시
- `city_names = []` : 빈 리스트 준비
- `lines = file.readlines()` : 파일의 모든 줄을 리스트로 읽기
- `line.split(':')[0]` : 한 줄을 `:` 기준으로 나눈 뒤 **첫 조각**(도시 이름 부분)만 선택
- `[:-1]` : 마지막 한 글자(이름 뒤의 공백)를 잘라 냄
- `city_names.append(city)` : 리스트에 추가
- `if city_related_to_question not in city_names:` : 드레스덴이 목록에 없으면 추가

> 🔰 예: `"Seoul : 대한민국의 수도"` → `split(':')` → `["Seoul ", " 대한민국의 수도"]` → `[0]` → `"Seoul "` → `[:-1]` → `"Seoul"`

In [17]:
# 14
### YOUR CODE HERE ###

city_name_path = r'C:\Users\RAG\1일차\실습 자료\Code\city_short.txt' #change this path

city_names = []

with open(city_name_path, 'r', encoding='utf-8') as file:
    lines = file.readlines()
    for line in lines:
        city = line.split(':')[0][:-1]
        city_names.append(city)

print(city_names) #100 different cities

if city_related_to_question not in city_names:
  city_names.append(city_related_to_question)

['New York City', 'London', 'Paris', 'Tokyo', 'Berlin', 'Shanghai', 'Los Angeles', 'Istanbul', 'Sydney', 'Seoul', 'Saint Petersburg', 'Mumbai', 'Beijing', 'Leipzig', 'Toronto', 'Rio de Janeiro', 'Mexico City', 'Budapest', 'Barcelona', 'Moscow', 'Vancouver', 'Dublin', 'Antananarivo', 'Suwon']


### 💡 [# 15] 코드 설명 — 위키백과 문서 불러오기

**한 줄 요약:** 도시 이름마다 위키백과 페이지를 읽어 `documents` 로 만듭니다.

- `wikipedia.set_user_agent("...")` : 위키백과 서버에 "누가 접속하는지" 알리는 이름표. 이것이 없으면 접속이 거부될 수 있습니다.
- `reader = WikipediaReader()` : 위키백과 읽기 도구 준비
- `reader.load_data(city_names, auto_suggest=False)` : 각 이름의 페이지를 읽어 Document 목록 생성. `auto_suggest=False` 는 "비슷한 다른 제목으로 멋대로 바꾸지 말고 정확히 그 제목만 찾으라"는 뜻입니다.

> 🔰 100여 개 페이지를 받아오므로 **몇 분** 걸릴 수 있습니다.

In [18]:
# 15
### YOUR CODE HERE ###
import wikipedia
wikipedia.set_user_agent("my-rag-project (test@example.com)")
reader = WikipediaReader()
documents = reader.load_data(city_names, auto_suggest=False)

<img src="https://i.gifer.com/B6Qs.gif" width="150">

This process will take some time.

If there was no problem with progressing this far, let's insert the Wikipedia page information transformed into the `Document` object.


### 💡 [# 16] 코드 설명 — 인덱스 만들기

**한 줄 요약:** 위키백과 문서들을 조각내고 임베딩해 검색용 인덱스를 만듭니다.

- `VectorStoreIndex.from_documents(documents)` : 조각내기 → 임베딩 → 저장을 한 번에 수행

> 🔰 문서 양이 많아 시간이 걸리고 임베딩 API 비용이 발생합니다.

In [19]:
# 16
### YOUR CODE HERE ###

index = VectorStoreIndex.from_documents(documents)

### 2. Check Query Engine

Make query engine from index, and check whether Wikipedia page information has been properly inserted using city-related queries.

We will using query related to Berlin which is element of `city_names`, and the answer of the query can be inferred from the information on the Wikipedia page.  

<br/>

####**Using query: 'What's the arts and culture scene in Berlin?'**

<img src = "https://i.imgur.com/RQkoLL1.png" width = 800>


### 💡 [# 17] 코드 설명 — Query Engine 동작 확인

**한 줄 요약:** 인덱스로 Query Engine을 만들고 "2024년 신장 지진 규모"를 물어봅니다.

- `index.as_query_engine()` : 검색+답변 창구 생성
- `textwrap.fill(str(response), 100)` : 답을 글자로 바꿔 100자 폭으로 줄바꿈

> 🔰 LLM이 모르는 최신 사건이라도, 위키백과 문서에 내용이 있으면 답할 수 있습니다.

In [25]:
# 17
### YOUR CODE HERE ###

query_engine = index.as_query_engine()
response = query_engine.query("What is the magnitude of the earthquake that occurred in Xinjiang, China in 2024?")
print(textwrap.fill(str(response), 100))

The magnitude of the earthquake that occurred in Xinjiang, China in 2024 is not provided in the
context information.


#### Let's see what happens to the response of query engine when get query asking for knowledge that does not exist in the database.

#### **Citiese and question-answer pairs that are not in 'city.txt'**

**Xinjiang**  
*Question:* What is the magnitude of the earthquake that occurred in Xinjiang, China in 2024?  
*Answer:* 7.1

**Jaipur**  
*Question:* Where in Jaipur is the proposed location for the third largest cricket stadium in the world, which can accommodate about 75,000 people?  
*Answer:* Chonp Village.


**Evaluate for yourself whether the response is accurate of inaccurate**

### 💡 [# 18] 코드 설명 — LLM이 모르는 질문 두 개 테스트

**한 줄 요약:** LLM 단독으로는 답하기 어려운 질문 2개(신장 지진, 자이푸르 크리켓 경기장)를 Query Engine에 묻습니다.

- `response_xinjiang.response` : 응답 객체 안의 **답변 글** 부분
- `pretty_print(...)` : # 5에서 만든 줄바꿈 함수
- 두 답 사이에 구분선을 출력합니다.

In [26]:
# 18
### YOUR CODE HERE ###

unknown_question_xinjiang = "What is the magnitude of the earthquake that occurred in Xinjiang, China in 2024?"
unknown_question_jaipur = "Where in Jaipur is the proposed location for the third largest cricket stadium in the world, which can accommodate about 75,000 people?"

response_xinjiang = query_engine.query(unknown_question_xinjiang)
response_jaipur = query_engine.query(unknown_question_jaipur)

print(pretty_print(response_xinjiang.response))
print("\n-------------------------\n")
print(pretty_print(response_jaipur.response))

The magnitude of the earthquake that occurred in Xinjiang, China in 2024 is not
provided in the context information.

-------------------------

Jaipur is the proposed location for the third largest cricket stadium in the
world, which can accommodate about 75,000 people.


### 3. Connect Retriever and Generator

Build a RAG from scratch.

Since RAG consists of two steps: retrieve and generation, it is a good choice to generate it as a class with two methods: `retrieve` and `generate_response`.

In addition, when we add a method that puts the output from `retrieve` method as the input of the `generate_response` method, the RAG class can be completed.

First, import libraries


### 💡 [# 19] 코드 설명 — OpenAI 클라이언트 준비

**한 줄 요약:** OpenAI 서버와 통신할 "창구 객체"를 만듭니다.

- `from openai import OpenAI` : OpenAI 클라이언트 클래스 불러오기
- `oai_client = OpenAI()` : 객체 생성. API 키는 환경변수에서 자동으로 읽습니다.

In [27]:
# 19
### YOUR CODE HERE ###

from openai import OpenAI

oai_client = OpenAI()

Next, make a RAG from scratch.


### 💡 [# 20] 코드 설명 — RAG를 클래스로 직접 만들기

**한 줄 요약:** "검색 → 답변 생성"을 하나로 묶은 `RAG_from_scratch` 클래스를 정의하고 객체를 만듭니다.

- `class RAG_from_scratch:` : 클래스(설계도) 정의. 안에 함수(메서드) 3개가 있습니다.
- `def retrieve(self, query: str) -> list:` : **검색 단계.** `query: str` 은 "query는 문자열"이라는 힌트, `-> list` 는 "리스트를 돌려준다"는 힌트입니다(힌트일 뿐 강제는 아님).
  - `query_engine.query(query)` : # 17에서 만든 Query Engine으로 관련 내용을 가져옵니다.
- `"""..."""` (함수 바로 아래) : 함수 설명서(**docstring**)
- `def generate_response(self, query, context_str):` : **생성 단계.** 검색된 내용(`context_str`)과 질문을 프롬프트에 넣어 LLM을 호출합니다.
  - `f"...{context_str}..."` : f-string으로 문맥과 질문을 끼워 넣음
  - `temperature=0` : 매번 일관된 답
  - `.choices[0].message.content` : 답변 글만 추출
- `def query(self, query):` : 위 두 단계를 차례로 실행하는 **전체 흐름**
- `rag = RAG_from_scratch()` : 설계도로 실제 객체 생성

> 🔰 여기서 `retrieve` 는 Query Engine을 통째로 부르므로, 실제로는 "검색된 원문"이 아니라 **Query Engine이 1차로 정리한 답**이 문맥으로 전달됩니다. 뒤(# 35)에서 이 구조를 개선합니다.

In [28]:
# 20
### YOUR CODE HERE ###

class RAG_from_scratch:
    def retrieve(self, query: str) -> list:
        """
        Retrieve relevant text from vector store.
        """
        results = query_engine.query(query)
        return results

    def generate_response(self, query: str, context_str: list) -> str:
        """
        Generate answer from context.
        """
        completion = oai_client.chat.completions.create(
        model="gpt-3.5-turbo",
        temperature=0,
        messages=
        [
            {"role": "user",
            "content":
            f"We have provided context information below. \n"
            f"---------------------\n"
            f"{context_str}"
            f"\n---------------------\n"
            f"Given this information, please answer the question: {query}"
            }
        ]
        ).choices[0].message.content
        return completion

    def query(self, query: str) -> str:
        context_str = self.retrieve(query)
        completion = self.generate_response(query, context_str)
        return completion

rag = RAG_from_scratch()

Now, it's time to remember the question we asked to LLM before making RAG.  
The default question was this.  

<br/>

Question: **Which Korean city has a relationship with Dresden?**  
Answer: **Daejeon**

<br/>

Let's use the RAG class we created to get this question to LLM. What will happen?  
Now that we've made the RAG class and the method in it, we can also see the results of query engine.

### 💡 [# 21] 코드 설명 — RAG로 드레스덴 질문 다시 풀기

**한 줄 요약:** # 12에서 LLM 단독으로 물었던 질문을 RAG에게 물어봅니다.

- `rag.query(city_question)` : 검색 → 생성 실행
- `print(pretty_print(answer))` : 줄바꿈해 출력

> 🔰 # 12의 답과 비교해 보세요. 위키백과 드레스덴 문서의 "자매도시" 정보를 근거로 답하면 성공입니다.

In [29]:
# 21
### YOUR CODE HERE ###

city_question = 'Which Korean city has a relationship with Dresden?'

answer = rag.query(city_question)

print(pretty_print(answer))

The Korean city of Daejeon has a relationship with Dresden.


Moreover, the query engine we use is directly generated from the index, and since retriever made of the `index.as _retriever()` method used in this query engine, we can also check the retrieved passages.

### 💡 [# 22] 코드 설명 — 어떤 조각이 검색되었는지 보기

**한 줄 요약:** 같은 질문에 대해 Retriever가 찾아온 조각의 원문을 출력합니다.

- `index.as_retriever()` : 검색 담당만 꺼내기
- `retriever.retrieve(city_question)` : 관련 조각 목록
- `f"Retrieved Paasge {i+1}\n"` : i는 0부터 시작하므로 사람이 보기 좋게 `+1` 해서 1번부터 표시

In [45]:
# 22
### YOUR CODE HERE ###

retriever = index.as_retriever()

ret = retriever.retrieve(city_question)
for i in range(len(ret)):
  print(f"Retrieved Paasge {i+1}\n", ret[i].text, '\n\n\n')

Retrieved Paasge 1
 === Friendly cities ===
Dresden also has friendly relations with:

Daejeon, South Korea
Gostyń, Poland
Shiraz, Iran


== Notable people ==


=== Public service ===
Augustus II the Strong (1670–1733), Elector of Saxony and King of Poland.
Augustus III of Poland (1696–1763), Elector of Saxony and King of Poland.
Frederick Augustus I of Saxony (1750–1827), King of Saxony.
Anthony of Saxony (1755–1836), King of Saxony
Count Heinrich von Bellegarde (1756–1845), Generalfeldmarschall and statesman.
Johann Adolf, Freiherr von Thielmann (1765–1824), Prussian cavalry soldier.
Wilhelm Adolf Becker (1796–1846), classical scholar. 



Retrieved Paasge 2
 The name of the city as well as the names of most of its boroughs and rivers are of Sorbian origin.
Dresden has a long history as the capital and royal residence for the Electors and Kings of Saxony, who for centuries furnished the city with cultural and artistic splendor, and was once by personal union the family seat of Polish

### II. Finding the best RAG App Configuration.

Let's change the configuration of the RAG and observe how its performance varies.

The performance of RAG is measured by `Groundedness`, `Answer Relevance`, and `Context Relevance`. However, `latency` and `total cost` are also important factors. Therefore, we shounld choose the evaluation metrics carefully and evaluate RAG's performance from various perspectives.

<br/>  

In this section, we'll modify the following factors to examine changes in RAG's performance:

#### 1. Chunk Size of Retriever: As the chunk size varies, retriever may or may not retrieve sufficient information. Let’s adjust the chunk size to see how the results change.
#### 2. Query Engine: Refine the prompt used in the Query Engine's answer generation to enhance the quality of its summaries.
#### 3. Prompt Design for RAG: Add clear instructions to the task so that the LLM generates the answer that the user is looking for.



First, let’s go through the process of varying the chunk size of the retriever.

When performing a search, the retriever finds and returns the most relevant information from the existing `nodes`. These nodes are determined by the `chunk size` and `chunk overlap` settings of the `SentenceSplitter`, which can be configured when creating the `index`.

Therefore, for questions where the information needed for the answer spans multiple chunks, the retriever may or may not return sufficient information depending on the chunk size.

This time, we will use the following two settings:


**<h4>Retriever_long - chunk size: 1024, chunk overlap: 200</h4>**

**<h4>Retriever_short - chunk size: 200, chunk overlap: 50</h4>**


### 💡 [# 23] 코드 설명 — 조각 크기가 다른 인덱스 두 개 만들기

**한 줄 요약:** 짧은 조각(200토큰)용 인덱스와 긴 조각(1024토큰)용 인덱스를 각각 만듭니다.

- `SentenceSplitter(chunk_size=200, chunk_overlap=50)` : 200토큰씩, 50토큰 겹치게 자름
- `SentenceSplitter(chunk_size=1024, chunk_overlap=200)` : 1024토큰씩, 200토큰 겹치게 자름
- `VectorStoreIndex.from_documents(documents=documents, transformations=[자르는도구])` : 지정한 방식으로 잘라 인덱스 생성

> 🔰 인덱스를 두 번 만들기 때문에 시간이 꽤 걸립니다. 특히 짧은 조각 쪽은 조각 수가 많아 더 오래 걸립니다.

In [31]:
# 23
### YOUR CODE HERE ###

text_splitter_short = SentenceSplitter(chunk_size=200, chunk_overlap=50)

index_short = VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter_short])

text_splitter_long = SentenceSplitter(chunk_size=1024, chunk_overlap=200)

index_long = VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter_long])

Additionally, to clearly compare the retrieved results, each retriever will be set to return only one node


### 💡 [# 24] 코드 설명 — 인덱스별 검색기 만들기

**한 줄 요약:** 두 인덱스에서 각각 "가장 비슷한 조각 1개만" 가져오는 검색기를 만듭니다.

- `as_retriever(similarity_top_k=1)` : `similarity_top_k` 는 **가져올 조각 개수**입니다. 1개로 맞춰 두 인덱스의 차이를 공정하게 비교합니다.

In [32]:
# 24
### YOUR CODE HERE ###

retriever_short = index_short.as_retriever(similarity_top_k=1)
retriever_long = index_long.as_retriever(similarity_top_k=1)

Now, let’s check the impact of `chunk size` by using the following question. This question is about Antananarivo, the capital of Madagascar.

To answer this question correctly, we need information on what the foundational materials of traditional architecture in Antananarivo are, as well as the efforts made to protect and restore this architectural heritage.

Let’s see how the retriever responds to this question with the code below.


### 💡 [# 25] 코드 설명 — 비교용 질문 준비

**한 줄 요약:** 안타나나리보의 전통 건축 재료와 문화유산 보호 노력에 대한, 답이 **길어야 하는** 질문을 저장합니다.

- 한 질문 안에 두 가지를 묻고 있어 넓은 문맥이 필요합니다.

In [33]:
# 25
### YOUR CODE HERE ###

question = "In Antananarivo, what were the foundational materials of traditional architecture, and what efforts have been made to protect and restore the city’s architectural and cultural heritage?"

### 💡 [# 26] 코드 설명 — 짧은 조각 인덱스의 검색 결과

**한 줄 요약:** 200토큰 인덱스에서 검색된 조각을 출력합니다.

- `retriever_short.retrieve(question)` : 검색
- 반복문으로 각 조각을 `pretty_print` 해 출력

> 🔰 조각이 짧아 질문의 일부에만 해당하는 내용이 나올 수 있습니다.

In [34]:
# 26
### YOUR CODE HERE ###

ret_passages_short = retriever_short.retrieve(question)
for i in range(len(ret_passages_short)):
  print(pretty_print(ret_passages_short[i].text))
  print("\n\n\n")

The plan, which is being implemented by the Institut des Métiers de la Ville,
prevents the destruction of historic buildings and other structures, and
establishes construction codes that ensure new structures follow historic
aesthetics. It also provides for awareness-raising campaigns in favor of
historic preservation and undertakes projects to restore dilapidated historic
buildings and sites. Under this plan, 19th-century sites, like the
Ambatondrafandrana tribunal and the second residence of Rainilaiarivony, have
been renovated.   == Demographics == Antananarivo has been the largest city on
the island since at least the late 18th century, when its population was
estimated at 15,000.






### 💡 [# 27] 코드 설명 — 긴 조각 인덱스의 검색 결과

**한 줄 요약:** 1024토큰 인덱스에서 검색된 조각을 출력합니다.

- # 26과 같은 코드에 검색기만 `retriever_long` 으로 바꿨습니다.

> 🔰 조각이 길어 더 많은 정보가 담기지만, 질문과 무관한 내용도 함께 들어옵니다. **짧으면 정밀, 길면 풍부** — 장단점이 있습니다.

In [35]:
# 27
### YOUR CODE HERE ###

ret_passages_long = retriever_long.retrieve(question)
for i in range(len(ret_passages_long)):
  print(pretty_print(ret_passages_long[i].text))
  print("\n\n\n")

This area is connected to Analakely by a tunnel constructed by the French in the
early 20th century. The tunnel opens toward Lake Anosy and the national Supreme
Court buildings and provides access to the residential neighborhood of
Mahamasina and its stadium. The bordering neighborhood of Anosy was developed in
the 1950s to house most of the national ministries and the Senate. Anosy, the
planned residential district of Soixante-Sept Hectares (often abbreviated to
"67") and the neighborhood of Isotry are among the city's most densely
populated, crime-ridden and impoverished neighborhoods. Approximately 40 percent
of inhabitants with electricity in their homes in the ville basse obtain it
illegally by splicing into city power lines. In these areas, houses are more
vulnerable to fires, flooding, and landslides, which are often triggered by the
annual cyclone season.   === Architecture ===  Before the mid-19th century, all
houses and marketplaces in Antananarivo, and throughout Madagascar,

Comparing the retrieved passages, you can see that when the `chunk size` is large, sufficient information is included, whereas with a small chunk size, the context is cut off.

By adjusting the `chunk size` in this way, the accuracy of the query engine can change. However, increasing the chunk size may also include unnecessary information, and detailed information might be overlooked by the generator.

Therefore, the optimal `chunk size` varies depending on the case, and this is an important configuration that can affect performance in RAG.

Choose the retriever and corresponding index you want to use for this practice. Later, you can modify this part and redeclare the query engine to change the type of retriever. These results will also impact the creation of the Query engine in the future

### 💡 [# 28] 코드 설명 — 사용할 구성 선택

**한 줄 요약:** 앞으로 쓸 인덱스(짧은 것/긴 것)와 검색 개수(2개)를 고릅니다.

- `retriever = index_short.as_retriever(similarity_top_k=2)` : 짧은 조각 인덱스에서 상위 2개 검색
- `#` 로 시작하는 줄은 **주석**(실행되지 않는 메모)입니다. 긴 인덱스를 쓰고 싶으면 위 줄 앞에 `#` 를 붙이고 아래 줄의 `#` 를 지우면 됩니다.
- `index = index_short` : 이후 코드가 쓸 `index` 를 지정

In [36]:
# 28
### YOUR CODE HERE ###

retriever = index_short.as_retriever(similarity_top_k=2)
# retriever = index_long.as_retriever(similarity_top_k=2)

index = index_short #If you want to use this, remove #
# index = index_long #If you want to use this, remove #

<h4>Secondly, we will redefine the query engine and response synthesizer.  
There are two options available:  </h4>

First, you can use the query engine to **generate a summary** instead of an answer. The downside of this approach is that important information may be lost during the summarization process.

The second option is to **modify the instructions of the current query engine** you have been using. Currently, the query engine is set to generate an answer directly, but you can change the prompt used for this as you see fit. However, if an error occurs in the query engine, it may propagate to the generator, causing error propagation.

As seen in previous experiments, crafting a good prompt significantly impacts the performance of an LLM. At this point, you will decide which version of the query engine to use and what prompt to employ.  

<br/>



### 💡 [# 29] 코드 설명 — 프롬프트 틀 두 가지 만들기

**한 줄 요약:** 맞춤 Query Engine에 쓸 부품을 불러오고, "답변용"과 "요약용" 프롬프트 틀을 만듭니다.

- import 6줄 : 맞춤 엔진의 부모 클래스, 검색기·합성기 타입, LLM, 프롬프트 틀
- `simple_qa_prompt` : 문맥만 보고 **질문에 답하라**는 틀. `{context_str}`, `{query_str}` 는 나중에 채워질 빈칸
- `short_sum_prompt` : 문맥을 **핵심 내용 위주로 요약하라**는 틀 (`{context_str}` 만 사용)

In [37]:
# 29
### YOUR CODE HERE ###

from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer
from llama_index.llms.openai import OpenAI
from llama_index.core import PromptTemplate

simple_qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)

short_sum_prompt = PromptTemplate(
"""Write a summary of the following. Try to use only the information provided.
Try to include as many key details as possible.
---------------------\n
{context_str}
---------------------\n
SUMMARY:"""
)

### 💡 [# 30] 코드 설명 — 맞춤 Query Engine 정의

**한 줄 요약:** 내가 고른 프롬프트로 LLM을 호출하는 `OurCustomQueryEngine` 클래스를 정의합니다.

- `class OurCustomQueryEngine(CustomQueryEngine):` : LlamaIndex의 기본 클래스를 **상속**
- `qa_prompt: PromptTemplate = simple_qa_prompt` : 프롬프트 부품(기본값은 답변용)
- `custom_query(self, query_str)` :
  - `self.retriever.retrieve(query_str)` : 조각 검색
  - `"\n\n".join([n.node.get_content() for n in nodes])` : 각 조각의 글을 꺼내 빈 줄로 이어 붙임 (`[... for n in nodes]` 는 한 줄 반복문)
  - `self.qa_prompt.format(...)` : 틀의 빈칸 채우기
  - `self.llm.complete(...)` : LLM 호출
- `llm = OpenAI(model="gpt-3.5-turbo")` : 사용할 LLM 준비

In [38]:
# 30
### YOUR CODE HERE ###

class OurCustomQueryEngine(CustomQueryEngine):

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: OpenAI
    qa_prompt: PromptTemplate = simple_qa_prompt

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)

        context_str = "\n\n".join([n.node.get_content() for n in nodes])
        response = self.llm.complete(
            self.qa_prompt.format(context_str=context_str, query_str=query_str)
        )

        return str(response)

llm = OpenAI(model="gpt-3.5-turbo")

You can directly see how the two versions of the query engine generate results by checking their outputs with the code below.



### 💡 [# 31] 코드 설명 — "답변형" 엔진 만들기

**한 줄 요약:** 답변용 프롬프트를 넣은 엔진을 만들어 베를린의 예술·문화에 대해 질문합니다.

- `retriever = index.as_retriever()` : 검색기 (※ # 28에서 정한 `similarity_top_k=2` 검색기를 기본값 검색기로 **덮어씁니다**)
- `get_response_synthesizer(response_mode="compact")` : 합성기(선언상 필요)
- `OurCustomQueryEngine(..., qa_prompt=simple_qa_prompt)` : 답변형 엔진 생성
- `res_answer = ...query("...")` : 결과 저장(출력은 # 33에서)

In [39]:
# 31
### YOUR CODE HERE ###

retriever = index.as_retriever()
synthesizer = get_response_synthesizer(response_mode="compact")

query_engine_answer = OurCustomQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    llm=llm,
    qa_prompt=simple_qa_prompt,
)

res_answer = query_engine_answer.query("What's the arts and culture scene in Berlin?")

### 💡 [# 32] 코드 설명 — "요약형" 엔진 만들기

**한 줄 요약:** 같은 부품에 프롬프트만 요약용으로 바꾼 엔진을 만들어 같은 질문을 합니다.

- `qa_prompt=short_sum_prompt` 만 다릅니다.
- `res_summary` 에 결과 저장

In [40]:
# 32
### YOUR CODE HERE ###

query_engine_sum = OurCustomQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    llm=llm,
    qa_prompt=short_sum_prompt,
)

res_summary = query_engine_sum.query("What's the arts and culture scene in Berlin?")

### 💡 [# 33] 코드 설명 — 두 엔진 결과 비교

**한 줄 요약:** 요약형 결과와 답변형 결과를 차례로 출력해 비교합니다.

> 🔰 검색된 조각은 같아도 **프롬프트에 따라 결과의 형태가 달라진다**는 것을 확인하세요.

In [41]:
# 33
### YOUR CODE HERE ###

print(res_summary)
print(res_answer)

Berlin is known for its vibrant street art culture, with the East Side Gallery along the Spree river in Friedrichshain being a popular attraction. The city also boasts contemporary art galleries such as KW Institute for Contemporary Art and KOW in Mitte, as well as nightlife hotspots like SO36 in Kreuzberg. The Kulturforum houses cultural institutions like the Gemäldegalerie and the Berliner Philharmonie, while the area around Hackescher Markt is known for its fashionable culture and Jewish heritage. The Straße des 17. Juni serves as a central east-west axis in the city, commemorating historical events in East Berlin.
Berlin has a rich arts and culture scene, with a strong emphasis on contemporary art. There are galleries such as KW Institute for Contemporary Art, KOW, Sprüth Magers, Blain Southern, Esther Schipper, Future Gallery, and König Gallerie. The city is also known for its vibrant street art culture, with the East Side Gallery along the Spree river being a notable example. In 

Now, before redifining RAG, you need to first decide which query engine to use


### 💡 [# 34] 코드 설명 — 사용할 엔진 선택

**한 줄 요약:** 최종 RAG에 쓸 Query Engine으로 요약형을 선택합니다.

- `query_engine = query_engine_sum` : 요약형 사용. 답변형을 쓰려면 주석(`#`)을 서로 바꾸면 됩니다.

In [42]:
# 34
### YOUR CODE HERE ###

query_engine = query_engine_sum #If you want to use this, remove #
# query_engine = query_engine_answer #If you want to use this, remove #

Now, let's create a new RAG using the query engine and retriever we've built above.  

In this process, we'll provide clearer instructions in the input prompt that goes to the RAG's generator.


### 💡 [# 35] 코드 설명 — 개선된 RAG 클래스

**한 줄 요약:** 검색 결과와 요약을 함께 활용하고, 더 꼼꼼한 프롬프트로 답을 만드는 `Refine_RAG` 를 정의합니다.

- `retrieve(self, query)` : 두 가지를 돌려줍니다.
  - `ret` : 검색된 조각 원문 목록
  - `results` : Query Engine(요약형)의 결과
  - `return ret, results` : 값 두 개를 한꺼번에 돌려주기(**튜플**)
- `generate_response(self, query, context_str)` :
  - `system` 메시지 : "간결하게 답하라"
  - `user` 메시지 : **지시사항 / 질문 / 제공된 문맥 / 답 / 이유** 형식으로 구성. 문맥이 틀리거나 무관하면 LLM 자신의 지식을 써도 된다고 허용하고, 답의 **이유**도 설명하게 합니다.
  - `oai_client.chat.completions.create(...)` : LLM 호출
- `query(self, query)` :
  - `ret, context_str = self.retrieve(query)` : 돌려받은 두 값을 각각의 변수에 나눠 담기
  - `#` 로 막아 둔 `print` 줄 : 중간 결과(검색 조각, 요약)를 보고 싶을 때 `#` 를 지워 사용
- `refine_rag = Refine_RAG()` : 객체 생성

In [43]:
# 35
### YOUR CODE HERE ###

class Refine_RAG:
    def retrieve(self, query: str) -> list:
        ret = retriever.retrieve(query)
        results = query_engine.query(query)
        return ret, results

    def generate_response(self, query: str, context_str: list) -> str:
        """
        Generate answer from context.
        """
        messages = [
            {
                "role": "system",
                "content": f"You are a helpful assistant. Answer as concisely as possible.",
            },
            {
                "role": "user",
                "content":
                    f"""
                    ###Instruction
Please answer the following question based on the provided context. Your answer should be short and concise.
Basically, you have to answer the question based on the provided context. But you can use your parametrized knowledge when the provided context was wrong or unrelated to the quesion.
When you generate the answer, you should explain the reason why you deduced such a answer from the context.

### Question
{query}

### Provided Context
{context_str}

### Answer


### Reason
                    """
            }
        ]

        response = oai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            messages=messages,
        )

        return response.choices[0].message.content

    def query(self, query: str) -> str:
        ret, context_str = self.retrieve(query)
        # for i in range(len(ret)):
        #   print("Retrieved Context: \n", ret[i].text) # use only when you want to see intermeidate result
        # print("\n\nIntermediate Summary: \n",  context_str) # use only when you want to see intermeidate result
        completion = self.generate_response(query, context_str)
        return completion

refine_rag = Refine_RAG()

Within the RAG class, you can inspect all values and content that are created and passed along.


### 💡 [# 36] 코드 설명 — 개선된 RAG 테스트

**한 줄 요약:** "수원시 의회가 생활폐기물 불법투기에 어떻게 대응했는가?"를 물어 결과를 봅니다.

- `refine_rag.query(sample_question)` : 검색 → 요약 → 최종 답변
- `print(f"\n\n{answer}")` : 빈 줄 두 개 뒤에 답 출력

> 🔰 답과 함께 "Reason(이유)"이 나오는지 확인하세요. # 28(조각 크기), # 34(엔진 종류)를 바꿔 가며 결과를 비교하면 좋은 연습이 됩니다.

In [44]:
# 36
### YOUR CODE HERE ###

sample_question = "City council of Suwon addressed illegal dumping of household waste in what way?"

answer = refine_rag.query(sample_question)

print(f"\n\n{answer}")



The city council of Suwon addressed illegal dumping of household waste by increasing urban greenery, which has helped reduce the scale of the problem.


I hope there are some improvement of RAG performance, even a little :)  

Of course, we can enhance the performance of the RAG not only through the three elements mentioned above but also in many other ways.

For instance, we can expect changes in performance by altering the following factors.

**Temperature**: This parameter controls the randomness of the output from a language model. A higher temperature increases the randomness, resulting in more varied and unpredictable text. Conversely, a lower temperature makes the model's responses more deterministic and predictable, often sticking closer to the most likely outcomes.  

**Top-p**: This parameter helps in controlling the diversity of the model's responses by focusing on the most probable tokens. Top-p sets a threshold to consider the top tokens up to the point where their cumulative probability exceeds a specified value p. This way, the model filters out less likely tokens and focuses only on a subset of the most probable tokens, which helps in generating coherent and contextually relevant outputs.

```Python
        response = oai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0, #change this value
            top_p=0, #change this value
            messages=messages,
        )
```

**Different Embedding Function**: Using more advanced embedding models could capture the meanings of tokens more effectively, potentially allowing for more accurate retrieval.

Increasing the dimensionality of embedding vectors can enable them to capture more information, thus improving their ability to find semantically similar passages. However, this might also lead to overfitting and make it difficult to measure similarities due to diluted data distribution across dimensions, i.e., curse of dimensionality.

Conversely, reducing the dimensions of vectors can increase computational efficiency and prevent overfitting. Yet, overly small dimensions may not capture enough information, leading to degraded performance.

```Python
from llama_index.core import Settings

# Option 1. global default
Settings.embed_model = OpenAIEmbedding()

# Option 2. local model
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5" #change this to the local path or huggingface model name
)

#per-index
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)
```

Therefore, by using one of the above codes, you can adjust the embedding function and the dimension of the embedding vector to improve the performance of the RAG


### 💡 [# 37] 코드 설명 — 빈 셀

번호만 있는 **연습용 빈 셀**입니다. 질문을 바꾸거나 설정을 바꿔 자유롭게 실험해 보세요.

In [ ]:
# 37
